# D3-04 Inventory flow assessment I — tracing intermediates

Conventional LCIA characterizes **biosphere flows** such as emissions and resource extraction. In this notebook, we use `edges` to count selected **technosphere exchanges** instead.

Our worked question is:

> How do the direct grid-electricity inputs of two hydrogen routes compare with the electricity-production exchanges traced through their supply chains?

We then reuse the same workflow for four material-related tracers. Run this notebook in the `bw` environment after the Day 1 BAFU import.

## Learning goals

By the end of the notebook, you should be able to:

- explain what a technosphere tracer counts;
- inspect a simple `CF = 1` method before using it;
- run one tracer method for several comparable activities;
- distinguish a direct foreground exchange from a full supply-chain tracer score;
- group traced exchanges into an interpretable result;
- recognize the limits of name-based tracer methods.

## Route through the notebook

1. Select two comparable hydrogen activities.
2. Inspect their direct grid-electricity inputs.
3. Read the electricity tracer rule.
4. Trace matched upstream exchanges.
5. Interpret two focused plots and one Sankey.
6. Transfer the workflow to material-related intermediates.

## What does a tracer score mean?

The usual Brightway calculation first solves the supply chain needed for the functional unit. A tracer then:

1. looks through the scaled technosphere exchanges;
2. keeps exchanges matching a rule;
3. multiplies each matched amount by `CF = 1`;
4. sums the matched amounts.

The result is therefore an **exchange-throughput indicator**, not an environmental impact score.

> **Important:** the same physical chain can contain several matched exchanges at different stages. A tracer score can therefore exceed the direct foreground input. Do not interpret it as additional energy use or as a mass/energy balance.

<details>
<summary>Optional matrix notation</summary>

For demand vector \(f\), the supply array is \(s=A^{-1}f\). Scaling the input entries of the technology matrix by \(s\) gives the technosphere exchange amounts required by the functional unit. `edges` applies an exchange-specific matrix \(E\); with matched values equal to one, summing \(E \circ T\) gives the traced throughput.

</details>

## 1) Set up the course project

In [ ]:
import json
from pathlib import Path

import bw2data as bd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from edges import EdgeLCIA, SupplyChain

plt.style.use("seaborn-v0_8-whitegrid")

SYSTEM_COLORS = {
    "PEM electrolysis": "#0F766E",
    "SOEC electrolysis": "#EA580C",
}

In [ ]:
bd.projects.set_current("aalborg-rlcia-2026")

if "bafu" not in bd.databases:
    raise ValueError("The 'bafu' database is missing. Complete the Day 1 import first.")

bafu = bd.Database("bafu")

assets_dir = Path("assets/d3-04")
if not assets_dir.exists():
    assets_dir = Path("tutorials/DAY 3/assets/d3-04")

if not assets_dir.exists():
    raise FileNotFoundError("Could not find the D3-04 assets folder.")

### Select activities safely

We match the activity name, reference product, and location. The helper stops if the database returns zero or several candidates.

In [ ]:
def find_activity(database, name, location):
    matches = [
        activity
        for activity in database
        if activity.get("name") == name
        and activity.get("reference product") == name
        and activity.get("location") == location
    ]

    if len(matches) != 1:
        raise ValueError(f"Expected one match for {name!r} in {location}; found {len(matches)}.")

    return matches[0]

In [ ]:
hydrogen_names = {
    "PEM electrolysis": (
        "Hydrogen production, gaseous, 30 bar, from PEM electrolysis, "
        "from grid electricity"
    ),
    "SOEC electrolysis": (
        "Hydrogen production, gaseous, 1 bar, from SOEC electrolysis, "
        "from grid electricity"
    ),
}

hydrogen_activities = {
    label: find_activity(bafu, name, "CH")
    for label, name in hydrogen_names.items()
}

for label, activity in hydrogen_activities.items():
    print(f"{label}: 1 {activity['unit']} of hydrogen, {activity['location']}")

## 2) Inspect the direct foreground electricity

Before following the supply chain, inspect the electricity exchange written directly in each hydrogen dataset.

These amounts answer: **how much grid electricity does the foreground process request?**

In [ ]:
direct_electricity = {}

for label, activity in hydrogen_activities.items():
    matches = [
        exchange
        for exchange in activity.technosphere()
        if exchange.input.get("name") == "Electricity, low voltage, at grid"
        and exchange.input.get("unit") == "kilowatt hour"
    ]

    if len(matches) != 1:
        raise ValueError(f"Expected one direct grid-electricity exchange for {label}.")

    direct_electricity[label] = float(matches[0]["amount"])
    print(f"{label}: {direct_electricity[label]:.1f} kWh / kg H₂")

## 3) Inspect the tracer method

The electricity tracer keeps suppliers whose reference product starts with `Electricity`, whose unit is `kilowatt hour`, and whose name does not contain one of four excluded terms.

Because the direct supplier is named `Electricity, low voltage, at grid`, it is excluded by `voltage`. The tracer score and direct foreground amount are therefore **two related measures, not two additive components**.

In [ ]:
electricity_method_path = assets_dir / "tracked_electricity_method.json"
electricity_method = json.loads(electricity_method_path.read_text())
electricity_rule = electricity_method["exchanges"][0]["supplier"]

print(f"Method: {electricity_method['name']}")
print(f"Reported unit: {electricity_method['unit']}")
print(
    "Match:",
    f"reference product {electricity_rule['operator']} "
    f"{electricity_rule['reference product']!r}",
)
print("Required unit:", electricity_rule["unit"])
print("Excluded terms:", ", ".join(electricity_rule["excludes"]))

## 4) Run the tracer

The helper below keeps the repeated `edges` workflow in one place:

`LCI → apply method strategies → evaluate CFs → LCIA → matched rows`

It removes zero-impact rows and checks that the remaining rows reconcile with the reported score.

In [ ]:
def run_tracer(activity, method_path):
    lca = EdgeLCIA(demand={activity: 1}, method=str(method_path))
    lca.lci()
    lca.apply_strategies()
    lca.evaluate_cfs()
    lca.lcia()

    table = lca.generate_cf_table(include_unmatched=False).copy()
    table = table.loc[table["impact"].abs() > 1e-12].copy()

    if not np.isclose(table["impact"].sum(), lca.score, rtol=1e-6, atol=1e-9):
        raise ValueError("The matched rows do not reconcile with the tracer score.")

    return float(lca.score), table

In [ ]:
electricity_results = {}

for label, activity in hydrogen_activities.items():
    score, table = run_tracer(activity, electricity_method_path)
    electricity_results[label] = {
        "score": score,
        "table": table,
    }

comparison = pd.DataFrame(
    [
        {
            "system": label,
            "direct grid input [kWh/kg H2]": direct_electricity[label],
            "traced upstream throughput [kWh/kg H2]": result["score"],
            "non-zero matched exchanges": len(result["table"]),
        }
        for label, result in electricity_results.items()
    ]
)

### Direct input versus traced throughput

The connected dots compare the two measures without implying that one is a component of the other.

In [ ]:
systems = list(hydrogen_activities)
fig, ax = plt.subplots(figsize=(9.2, 3.5))

for y, label in enumerate(systems):
    direct = direct_electricity[label]
    traced = electricity_results[label]["score"]
    color = SYSTEM_COLORS[label]

    ax.plot([direct, traced], [y, y], color="#CBD5E1", linewidth=4, zorder=1)
    ax.scatter(
        direct, y, s=110, facecolor="white", edgecolor=color, linewidth=2.5,
        label="Direct grid input" if y == 0 else None, zorder=2,
    )
    ax.scatter(
        traced, y, s=110, color=color,
        label="Matched upstream throughput" if y == 0 else None, zorder=2,
    )
    label_offset = -14 if y == 0 else 14
    ax.annotate(
        f"{direct:.1f}", (direct, y), xytext=(0, label_offset),
        textcoords="offset points", ha="center", va="center", color=color,
    )
    ax.annotate(
        f"{traced:.1f}", (traced, y), xytext=(0, label_offset),
        textcoords="offset points", ha="center", va="center", color=color,
    )

ax.set_yticks(range(len(systems)), systems)
ax.invert_yaxis()
ax.set_xlabel("Electricity exchange amount [kWh / kg H2]")
ax.set_title("Direct grid input versus matched upstream throughput")
ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5))
ax.grid(axis="x", alpha=0.25)
ax.grid(axis="y", visible=False)
ax.spines[["top", "right", "left"]].set_visible(False)
plt.tight_layout()
plt.show()

### Where are the matched suppliers?

Instead of listing thousands of matched rows, aggregate their contributions by supplier location. The five largest locations are shown separately; the rest are grouped as `Other`.

In [ ]:
location_parts = []

for label, result in electricity_results.items():
    grouped = (
        result["table"]
        .assign(system=label)
        .groupby(["system", "supplier location"], dropna=False, as_index=False)["impact"]
        .sum()
    )
    location_parts.append(grouped)

location_scores = pd.concat(location_parts, ignore_index=True)
location_scores["supplier location"] = location_scores["supplier location"].fillna("Unknown")

top_locations = (
    location_scores.groupby("supplier location")["impact"]
    .sum()
    .nlargest(5)
    .index
)

location_scores["location group"] = location_scores["supplier location"].where(
    location_scores["supplier location"].isin(top_locations),
    "Other",
)

In [ ]:
location_matrix = (
    location_scores
    .pivot_table(
        index="system",
        columns="location group",
        values="impact",
        aggfunc="sum",
        fill_value=0,
    )
    .reindex(systems)
)

location_order = [loc for loc in top_locations if loc in location_matrix] + ["Other"]
location_matrix = location_matrix.reindex(columns=location_order, fill_value=0)

fig, ax = plt.subplots(figsize=(9.3, 3.8))
left = np.zeros(len(systems))
colors = plt.cm.Set2(np.linspace(0, 1, len(location_order)))

for location, color in zip(location_order, colors):
    values = location_matrix[location].to_numpy()
    ax.barh(systems, values, left=left, label=location, color=color)
    left += values

for y, total in enumerate(left):
    ax.text(total + 0.6, y, f"{total:.1f}", va="center")

ax.invert_yaxis()
ax.set_xlim(0, left.max() * 1.12)
ax.set_xlabel("Traced upstream throughput [kWh / kg H2]")
ax.set_title("Supplier locations behind the matched electricity exchanges")
ax.legend(title="Location", frameon=False, loc="upper left",
          bbox_to_anchor=(1.02, 1.0))
ax.grid(axis="x", alpha=0.25)
ax.grid(axis="y", visible=False)
ax.spines[["top", "right", "left"]].set_visible(False)
plt.tight_layout()
plt.show()

### Quick checkpoint

Before running the next cell, answer:

1. Which route is lower for both the direct grid input and traced throughput?
2. Are the two traced/direct ratios substantially different?
3. Why would it be wrong to call the traced score an environmental impact?

The short answer scaffold below extracts only the values needed to check your interpretation.

In [ ]:
ranked = comparison.sort_values(
    "traced upstream throughput [kWh/kg H2]"
).reset_index(drop=True)

lower_route = ranked.loc[0, "system"]
ratios = (
    comparison.set_index("system")["traced upstream throughput [kWh/kg H2]"]
    / comparison.set_index("system")["direct grid input [kWh/kg H2]"]
)

print(f"Lower traced throughput: {lower_route}")
for label in systems:
    print(f"{label}: traced/direct ratio = {ratios[label]:.2f}")
print("The method counts inventory exchanges with CF = 1; it does not model damage.")

## 5) Explain the PEM result with one Sankey

The location plot tells us **where** matched suppliers are located. The Sankey adds the supply-chain sequence.

To keep it readable, we merge repeated node instances, stop after five levels, and group branches contributing less than 1% of the total.

In [ ]:
pem_chain = SupplyChain(
    activity=hydrogen_activities["PEM electrolysis"],
    method=str(electricity_method_path),
    amount=1,
    level=5,
    cutoff=0.01,
    cutoff_basis="total",
    collapse_markets=True,
)

pem_chain.bootstrap()
pem_chain_table, _, _ = pem_chain.calculate()

print(f"Rows retained for the Sankey: {len(pem_chain_table)}")

In [ ]:
pem_sankey = pem_chain.plot_sankey(
    pem_chain_table,
    width_max=1250,
    height_max=700,
    node_instance_mode="merge",
    node_thickness=14,
    node_pad=12,
)

pem_sankey

## 6) Transfer the workflow to material-related intermediates

We now reuse the same helper with four name-based tracer methods:

- wood;
- steel;
- aluminium;
- concrete.

Each method matches kilogram exchanges whose reference product contains the material keyword and assigns `CF = 1`.

These scores are **not bills of materials**: rules can miss unfamiliar names, and several exchanges along one supply chain can be counted. For the same reason, we do not normalize the four indicators into a 100% “composition.”

In [ ]:
material_method_paths = {
    "Wood": assets_dir / "tracked_wood_method.json",
    "Steel": assets_dir / "tracked_steel_method.json",
    "Aluminium": assets_dir / "tracked_aluminium_method.json",
    "Concrete": assets_dir / "tracked_concrete_method.json",
}

for label, path in material_method_paths.items():
    rule = json.loads(path.read_text())["exchanges"][0]["supplier"]
    print(f"{label}: reference product contains {rule['reference product']!r}")

In [ ]:
material_rows = []

for system, activity in hydrogen_activities.items():
    for material, path in material_method_paths.items():
        score, table = run_tracer(activity, path)
        material_rows.append(
            {
                "system": system,
                "material": material,
                "score": score,
                "non-zero matched exchanges": len(table),
            }
        )

material_scores = pd.DataFrame(material_rows)

The values span more than two orders of magnitude, so the dot plot uses a logarithmic x-axis. The labels still report the actual tracer scores.

In [ ]:
material_order = list(material_method_paths)
fig, ax = plt.subplots(figsize=(8.5, 4.6))
offsets = {"PEM electrolysis": -0.13, "SOEC electrolysis": 0.13}

for system in systems:
    subset = material_scores.set_index(["system", "material"]).loc[system]
    x = subset.loc[material_order, "score"].to_numpy()
    y = np.arange(len(material_order)) + offsets[system]

    ax.scatter(
        x, y, s=90, color=SYSTEM_COLORS[system], label=system, zorder=3,
    )
    for x_value, y_value in zip(x, y):
        value_label = f"{x_value:.4f}" if x_value < 0.01 else f"{x_value:.3f}"
        ax.annotate(
            value_label,
            (x_value, y_value),
            xytext=(6, 0),
            textcoords="offset points",
            va="center",
            fontsize=9,
        )

ax.set_xscale("log")
ax.set_yticks(range(len(material_order)), material_order)
ax.invert_yaxis()
ax.set_xlabel("Tracer score [kg traced / kg H2] — logarithmic scale")
ax.set_title("Name-based material tracer scores")
ax.legend(frameon=False)
ax.grid(axis="x", which="both", alpha=0.25)
ax.grid(axis="y", visible=False)
ax.spines[["top", "right", "left"]].set_visible(False)
plt.tight_layout()
plt.show()

### Interpretation

- PEM has higher wood and concrete tracer scores.
- SOEC has a higher steel tracer score.
- Aluminium is much smaller for both routes but remains visible on the logarithmic axis.

The result is useful for **screening where to investigate**, not for claiming exact material content.

## Recap and cautions

- A `CF = 1` technosphere method counts matched exchange throughput.
- The direct foreground exchange and the tracer score answer different questions.
- Aggregation and a focused Sankey are more useful than printing thousands of matched rows.
- Name-based rules must be audited; terminology and exclusions determine what is counted.
- Tracer indicators are not environmental impacts, bills of materials, or strict mass/energy balances.

### Optional extension

Choose **one** material method, inspect its JSON rule, and build a Sankey by replacing the method path in the PEM example. Avoid generating several Sankeys unless each answers a distinct interpretation question.

In D3-05, we keep the paired-technosphere logic but replace `CF = 1` with GeoPolRisk criticality factors.